# Residual Clustering (post-reclassification)

Clusters the low-confidence listings that remain after anchor refinement and reclassification, to identify:
1. **Missing NEW_L3 categories** — clusters that don't map cleanly to any existing anchor
2. **Anchor description gaps** — correctly-assigned items scoring low due to thin anchor wording

No new embeddings are generated — all vectors are retrieved from the existing two-volume cache.

**Input:** `artifacts/analysis/residual_reclassification/residual_reclassified.csv`  
*(Falls back to `artifacts/analysis/master_residual_for_clustering.csv` if reclassification has not been run yet.)*  
**Output:** `artifacts/analysis/l4_residual_clustering/cluster_results.csv` + sample printouts

In [ ]:
import sys
from pathlib import Path

# Resolve project root (two levels up from notebooks/)
PROJECT_ROOT = Path("__file__").resolve().parent.parent.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pickle
import numpy as np
import pandas as pd
from sklearn.cluster import MiniBatchKMeans
from sklearn.preprocessing import normalize
import matplotlib.pyplot as plt

from product_classifier_utils import (
    load_product_data,
    build_product_text,
    stable_text_hash,
    get_bedrock_client,
)

print("Imports OK")

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────

# Input: residual after anchor refinement + reclassification.
# Falls back automatically to the original master residual if reclassified file doesn't exist yet.
_RECLASSIFIED   = PROJECT_ROOT / "artifacts/analysis/residual_reclassification/residual_reclassified.csv"
_ORIGINAL       = PROJECT_ROOT / "artifacts/analysis/master_residual_for_clustering.csv"
RESIDUAL_CSV    = _RECLASSIFIED if _RECLASSIFIED.exists() else _ORIGINAL

# Embedding caches (v2 loads in-memory; v1 uses keys index to avoid 32GB OOM)
CACHE_V2_PATH   = PROJECT_ROOT / "artifacts/cache/embedding_cache_new.pkl"
CACHE_KEYS_PATH = PROJECT_ROOT / "artifacts/cache/embedding_cache_keys.pkl"
CACHE_V1_PATH   = PROJECT_ROOT / "artifacts/cache/embedding_cache.pkl"

# (Optional) Set True to reload PRODUCT_NAME from Snowflake for richer sample display.
# Requires an active AWS SSO session. Set False to skip and use cached text only.
RELOAD_FROM_SNOWFLAKE = False
PRODUCTS_TABLE        = "SNOWFLAKE_LEARNING_DB.SMCMAHON_PRODUCTS.NEW_L3_CLASSIFICATIONS"
AWS_PROFILE           = "staging.admin"

# Clustering
N_CLUSTERS         = 30    # Start here; adjust after reviewing elbow plot
RANDOM_STATE       = 42
SAMPLE_PER_CLUSTER = 15    # Listings to print per cluster for manual review

# True  → cluster within each L3 bucket (tighter, good for L5 discovery)
# False → cluster full residual together (catches missing L3s)
CLUSTER_PER_L4 = True

# LLM cluster labeling (optional — set True only if Claude/Bedrock access confirmed)
RUN_LLM_LABELING = False
LLM_MODEL_ID     = "us.anthropic.claude-3-haiku-20241022-v1:0"
LLM_MAX_CLUSTERS = 30
LLM_TEMPERATURE  = 0.2

# Output
OUTPUT_DIR = PROJECT_ROOT / "artifacts/analysis/l4_residual_clustering"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Residual CSV      : {RESIDUAL_CSV.name}")
print(f"  (using reclassified)" if RESIDUAL_CSV == _RECLASSIFIED else "  (using original — reclassify_residual.py not yet run)")
print(f"Cache v2          : {CACHE_V2_PATH}")
print(f"Cache v1 keys     : {CACHE_KEYS_PATH}")
print(f"N_CLUSTERS        : {N_CLUSTERS}")
print(f"Per-L3 mode       : {CLUSTER_PER_L4}")
print(f"Reload Snowflake  : {RELOAD_FROM_SNOWFLAKE}")
print(f"Output dir        : {OUTPUT_DIR}")

## 1. Load Residual IDs

In [ ]:
residual_meta = pd.read_csv(RESIDUAL_CSV)
residual_meta["PRODUCT_ID"] = residual_meta["PRODUCT_ID"].astype(str)
residual_ids  = set(residual_meta["PRODUCT_ID"])

# Support both old (ASSIGNED_L4_LABEL) and new (ASSIGNED_NEW_L3_LABEL) column names
if "ASSIGNED_NEW_L3_LABEL" in residual_meta.columns:
    LABEL_COL = "ASSIGNED_NEW_L3_LABEL"
elif "ASSIGNED_L4_LABEL" in residual_meta.columns:
    LABEL_COL = "ASSIGNED_L4_LABEL"
else:
    raise ValueError("Cannot find label column in residual CSV")

print(f"Residual listings : {len(residual_ids):,}")
print(f"Label column used : {LABEL_COL}")
print(f"Columns in CSV    : {list(residual_meta.columns)}")
print()
print("NEW_L3 breakdown of residual:")
print(residual_meta[LABEL_COL].value_counts().to_string())

## 2. Build Working DataFrame

The residual CSV already contains PRODUCT_ID, assigned labels, and confidence scores.
We build the hash used for cache lookup from the product text.

If `RELOAD_FROM_SNOWFLAKE = True`, PRODUCT_NAME is fetched from Snowflake for richer sample display.
Otherwise, we use only what's in the residual CSV (still sufficient for clustering).

In [ ]:
df = residual_meta.copy()

if RELOAD_FROM_SNOWFLAKE:
    print(f"Reloading PRODUCT_NAME from {PRODUCTS_TABLE} ...")
    raw_sf = load_product_data(table=PRODUCTS_TABLE, aws_profile=AWS_PROFILE, row_limit=None)
    raw_sf["PRODUCT_ID"] = raw_sf["PRODUCT_ID"].astype(str)
    name_cols = ["PRODUCT_ID", "PRODUCT_NAME"]
    available = [c for c in name_cols if c in raw_sf.columns]
    df = df.merge(raw_sf[available], on="PRODUCT_ID", how="left")
    print(f"PRODUCT_NAME merged from Snowflake for {df['PRODUCT_NAME'].notna().sum():,} rows")
else:
    if "PRODUCT_NAME" not in df.columns:
        df["PRODUCT_NAME"] = None
    print("Snowflake reload skipped. PRODUCT_NAME will be blank in sample display.")

# Build text for hash — used to look up embeddings in the cache
# The residual CSV may include PRODUCT_NAME / DESCRIPTION if it came from combine_and_publish
text_parts_available = [c for c in ["PRODUCT_NAME", "DESCRIPTION", "PRICING_STATUS_C", "LIST_PRICE_C"] if c in df.columns and df[c].notna().any()]
if len(text_parts_available) >= 2:
    df["PRODUCT_TEXT"] = build_product_text(df)
elif "PRODUCT_NAME" in df.columns:
    df["PRODUCT_TEXT"] = df["PRODUCT_NAME"].fillna("").str.strip()
else:
    raise ValueError("Not enough text columns to build product text for hash lookup. Set RELOAD_FROM_SNOWFLAKE=True.")

df["TEXT_HASH"] = df["PRODUCT_TEXT"].apply(stable_text_hash)
print(f"Working set: {len(df):,} rows")
print(f"Text columns used: {text_parts_available}")

## 3. Retrieve Embeddings from Two-Volume Cache

No Bedrock calls — vectors are pulled from the split cache (v2 first, then v1 key index).
The 32GB v1 cache is NOT loaded into memory; only v2 (13GB) is loaded directly.
Residuals whose hashes are only in v1 are flagged; run `reclassify_residual.py --phase extract`
outside the notebook if you need to cluster those listings too.

In [ ]:
print(f"Loading volume 2 cache ({CACHE_V2_PATH.stat().st_size/1e9:.1f} GB) ...")
with open(CACHE_V2_PATH, "rb") as f:
    cache_v2 = pickle.load(f)
print(f"Volume 2 entries: {len(cache_v2):,}")

print(f"Loading volume 1 key index ({CACHE_KEYS_PATH.stat().st_size/1e6:.0f} MB) ...")
with open(CACHE_KEYS_PATH, "rb") as f:
    cache_v1_keys = pickle.load(f)
print(f"Volume 1 keys: {len(cache_v1_keys):,}")

vectors, valid_mask, source_vol = [], [], []
for h in df["TEXT_HASH"]:
    if h in cache_v2:
        vectors.append(cache_v2[h])
        valid_mask.append(True)
        source_vol.append("v2")
    elif h in cache_v1_keys:
        # Vector is in the 32GB v1 cache — skipped here to avoid OOM.
        # These rows are excluded from clustering below.
        valid_mask.append(False)
        source_vol.append("v1_skip")
    else:
        valid_mask.append(False)
        source_vol.append("missing")

valid_mask = np.array(valid_mask)
df_valid   = df[valid_mask].copy().reset_index(drop=True)
X          = normalize(np.array(vectors), norm="l2")
del cache_v2

in_v1   = sum(1 for s in source_vol if s == "v1_skip")
missing = sum(1 for s in source_vol if s == "missing")
print(f"\nCache hits (v2, clustered) : {valid_mask.sum():,}")
print(f"In v1 only (skipped)       : {in_v1:,}")
print(f"Not in any cache           : {missing:,}")
print(f"Embedding dim              : {X.shape[1]}")
if in_v1 > 0:
    print(f"\nNOTE: {in_v1:,} residuals live in volume 1 (32GB). To include them:")
    print("  Run: python reclassify_residual.py --phase extract")
    print("  Then load the extracted .npy file manually if needed.")

## 4. Elbow Plot — Choose K

In [ ]:
k_range  = range(10, 51, 5)
inertias = []

print("Running elbow sweep (MiniBatchKMeans)...")
for k in k_range:
    km = MiniBatchKMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=3, batch_size=4096)
    km.fit(X)
    inertias.append(km.inertia_)
    print(f"  k={k:3d}  inertia={km.inertia_:,.0f}")

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(list(k_range), inertias, marker="o", color="steelblue")
ax.axvline(N_CLUSTERS, color="tomato", linestyle="--", label=f"Selected K={N_CLUSTERS}")
ax.set_xlabel("K")
ax.set_ylabel("Inertia")
ax.set_title("Elbow Plot — Residual Clustering")
ax.legend()
plt.tight_layout()
plt.show()

print(f"\nReview the elbow above, then update N_CLUSTERS in the config cell and re-run from Cell 5 onward.")

## 5. Cluster

In [ ]:
if CLUSTER_PER_L4:
    print(f"Clustering within each NEW_L3 bucket (up to {N_CLUSTERS} clusters each) ...")
    df_valid["CLUSTER"] = -1
    cluster_offset = 0
    for l3_label, grp_idx in df_valid.groupby(LABEL_COL).groups.items():
        grp_idx = list(grp_idx)
        grp_X = X[grp_idx]
        k = min(N_CLUSTERS, max(2, len(grp_idx) // 10))
        if len(grp_idx) < 10:
            df_valid.loc[grp_idx, "CLUSTER"] = cluster_offset
            cluster_offset += 1
            print(f"  {l3_label}: {len(grp_idx)} rows — single cluster (too small)")
            continue
        km = MiniBatchKMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=3, batch_size=2048)
        labels = km.fit_predict(grp_X)
        for i, idx in enumerate(grp_idx):
            df_valid.at[idx, "CLUSTER"] = labels[i] + cluster_offset
        cluster_offset += k
        print(f"  {l3_label}: {k} clusters ({len(grp_idx):,} listings)")
else:
    print(f"Clustering full residual into {N_CLUSTERS} clusters ...")
    km = MiniBatchKMeans(
        n_clusters=N_CLUSTERS,
        random_state=RANDOM_STATE,
        n_init=5,
        batch_size=4096,
        verbose=0,
    )
    df_valid["CLUSTER"] = km.fit_predict(X)

cluster_sizes = df_valid["CLUSTER"].value_counts().sort_index()
print(f"\nTotal clusters: {df_valid['CLUSTER'].nunique()}")
print(f"Cluster sizes (min={cluster_sizes.min():,}, max={cluster_sizes.max():,}, median={cluster_sizes.median():,.0f})")

## 6. Sample Listings per Cluster

For each cluster: show the dominant assigned-L4 label and sample product names.
Look for clusters where the dominant L4 looks **wrong** or **absent from the current anchor set** — those are candidates for a new L4.

In [ ]:
name_col = "PRODUCT_NAME" if ("PRODUCT_NAME" in df_valid.columns and df_valid["PRODUCT_NAME"].notna().any()) else "PRODUCT_TEXT"

for cluster_id in sorted(df_valid["CLUSTER"].unique()):
    grp = df_valid[df_valid["CLUSTER"] == cluster_id]
    dominant_l3  = grp[LABEL_COL].value_counts().idxmax()
    dominant_pct = grp[LABEL_COL].value_counts().max() / len(grp) * 100
    avg_conf     = grp["CONFIDENCE"].mean()

    print(f"\n{'='*70}")
    print(f"CLUSTER {cluster_id:3d} | {len(grp):6,} listings | NEW_L3: {dominant_l3} ({dominant_pct:.0f}%) | avg confidence: {avg_conf:.3f}")
    print(f"{'='*70}")

    sample = grp.sample(min(SAMPLE_PER_CLUSTER, len(grp)), random_state=RANDOM_STATE)
    for _, row in sample.iterrows():
        name = str(row.get(name_col, ""))[:90]
        src  = f"[{row['SOURCE']}]" if "SOURCE" in row.index else ""
        print(f"  {src} [{row['CONFIDENCE']:.3f}] {name}")

## 7. (Optional) LLM Cluster Labeling

Only runs if `RUN_LLM_LABELING = True` in the config cell and Claude access is available.

In [ ]:
cluster_labels = {}

if RUN_LLM_LABELING:
    import json
    import boto3

    bedrock = get_bedrock_client(profile_name=AWS_PROFILE, region="us-east-1")
    clusters_to_label = sorted(df_valid["CLUSTER"].unique())[:LLM_MAX_CLUSTERS]

    for cluster_id in clusters_to_label:
        grp = df_valid[df_valid["CLUSTER"] == cluster_id]
        samples = grp.sample(min(20, len(grp)), random_state=RANDOM_STATE)[name_col].tolist()
        samples_text = "\n".join(f"- {s[:100]}" for s in samples)

        prompt = (
            "You are a product taxonomy expert for a life science and lab supply marketplace.\n"
            "Below is a sample of product names from one cluster. Provide:\n"
            "1. A short label (3–6 words) for this cluster\n"
            "2. One sentence explaining what unifies these products\n"
            "3. Whether this fits an existing L4 category or might need a NEW one\n"
            "Existing L4s: Chemicals & Solvents, Antibodies, Proteins & Peptides, "
            "Equipment Instruments & Parts, Lab Supplies & Consumables, Molecular Biology Reagents, "
            "Kits & Assays, Cell & Tissue Culture, Biospecimens, Animal Models, "
            "Controls Calibrators & Standards, Furniture & Storage, General Office Supplies.\n\n"
            f"Products:\n{samples_text}\n\n"
            "Respond in JSON: {\"label\": \"...\", \"description\": \"...\", \"fits_existing_l4\": true/false, \"suggested_l4\": \"...\"}"
        )

        try:
            response = bedrock.invoke_model(
                modelId=LLM_MODEL_ID,
                body=json.dumps({
                    "anthropic_version": "bedrock-2023-05-31",
                    "max_tokens": 300,
                    "temperature": LLM_TEMPERATURE,
                    "messages": [{"role": "user", "content": prompt}],
                }),
                contentType="application/json",
                accept="application/json",
            )
            raw = json.loads(response["body"].read())["content"][0]["text"]
            parsed = json.loads(raw)
            cluster_labels[cluster_id] = parsed
            flag = "** NEW L4? **" if not parsed.get("fits_existing_l4", True) else ""
            print(f"Cluster {cluster_id:3d}: {parsed.get('label')} {flag}")
        except Exception as e:
            print(f"Cluster {cluster_id:3d}: LLM error — {e}")
            cluster_labels[cluster_id] = {"label": f"cluster_{cluster_id}", "error": str(e)}
else:
    print("LLM labeling skipped (RUN_LLM_LABELING=False).")
    print("Review the sample printouts in Cell 6 and manually assign labels below.")

## 8. Manual Label Assignment

After reviewing Cell 6 output, fill in the dictionary below.
Mark `new_l4=True` for any cluster that doesn't fit the existing 13 anchors.

In [ ]:
# Fill this in after reviewing cluster samples above.
# Format: cluster_id -> {"label": "...", "suggested_l4": "...", "new_l4": True/False}
manual_labels = {
    # 0: {"label": "Specialty Solvents",     "suggested_l4": "Chemicals & Solvents", "new_l4": False},
    # 1: {"label": "Surgical Instruments",   "suggested_l4": "NEW",                  "new_l4": True},
}

if manual_labels:
    df_valid["CLUSTER_LABEL"] = df_valid["CLUSTER"].map(
        {k: v["label"] for k, v in manual_labels.items()}
    )
    new_l4_candidates = {k: v for k, v in manual_labels.items() if v.get("new_l4")}
    if new_l4_candidates:
        print("*** POTENTIAL NEW L4 CATEGORIES FOUND ***")
        for cid, info in new_l4_candidates.items():
            size = (df_valid["CLUSTER"] == cid).sum()
            print(f"  Cluster {cid} ({size:,} listings): {info['label']}")
    else:
        print("All clusters map to existing L4s — anchor descriptions may need minor tweaks but no new L4 required.")
else:
    print("Fill in manual_labels dict after reviewing cluster samples.")

## 9. Save Cluster Assignments

In [ ]:
out_cols = ["PRODUCT_ID", LABEL_COL, "CONFIDENCE", "CLUSTER", "SOURCE"]
out_cols = [c for c in out_cols if c in df_valid.columns]
if "CLUSTER_LABEL" in df_valid.columns:
    out_cols.append("CLUSTER_LABEL")
if "PRODUCT_NAME" in df_valid.columns and df_valid["PRODUCT_NAME"].notna().any():
    out_cols.insert(1, "PRODUCT_NAME")

mode_tag  = "per_l3" if CLUSTER_PER_L4 else f"k{N_CLUSTERS}"
out_path  = OUTPUT_DIR / f"residual_clusters_{mode_tag}.csv"
df_valid[out_cols].to_csv(out_path, index=False)
print(f"Saved: {out_path} ({len(df_valid):,} rows)")

# Cluster summary
summary = df_valid.groupby("CLUSTER").agg(
    size=("PRODUCT_ID", "count"),
    avg_confidence=("CONFIDENCE", "mean"),
    dominant_l3=(LABEL_COL, lambda x: x.value_counts().idxmax()),
).sort_values("size", ascending=False)

if manual_labels:
    summary["manual_label"] = summary.index.map({k: v["label"] for k, v in manual_labels.items()})
    summary["new_l3"]       = summary.index.map({k: v.get("new_l4", False) for k, v in manual_labels.items()})

summary_path = OUTPUT_DIR / f"cluster_summary_{mode_tag}.csv"
summary.to_csv(summary_path)
print(f"Summary: {summary_path}")
print()
print(summary.to_string())